# TrafficVision-Transformer — Baseline CA
**Người 3** | UIT-ADrone Anomaly Detection

Thứ tự chạy: **Config → GPU check → Mount Drive → Clone → Install → Import check → Train → Inference → AUC → Visualize → Backup**

## ⚙️ 0. Cấu hình path — chỉ sửa ở đây

In [ ]:
# ==============================================================
# Chỉ cần sửa cell này, tất cả cell bên dưới tự dùng biến này
# ==============================================================

SEMINAR_DIR = '/content/drive/MyDrive/Seminar'   # thư mục gốc trên Drive
GITHUB_REPO = 'https://github.com/MinhCYB/TrafficVision-Transformer.git'

# ---- Tự động tính từ SEMINAR_DIR, không cần sửa ----
DATA_DIR    = f'{SEMINAR_DIR}/UIT-ADrone'
SAVE_DIR    = f'{SEMINAR_DIR}/TrafficVision/experiments_andt_ADrone_baseline_CA'
RESULTS_DIR = f'{SEMINAR_DIR}/TrafficVision/results'
REPO_DIR    = '/content/TrafficVision-Transformer'

print('SEMINAR_DIR :', SEMINAR_DIR)
print('DATA_DIR    :', DATA_DIR)
print('SAVE_DIR    :', SAVE_DIR)
print('REPO_DIR    :', REPO_DIR)

SEMINAR_DIR : /content/drive/MyDrive/Seminar
DATA_DIR    : /content/drive/MyDrive/Seminar/UIT-ADrone
SAVE_DIR    : /content/drive/MyDrive/Seminar/TrafficVision/experiments_andt_ADrone_baseline_CA
REPO_DIR    : /content/TrafficVision-Transformer


## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('Device        :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

Thu Jun  4 08:05:37 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             11W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
checks = {
    'Dataset train' : f'{DATA_DIR}/train/frames',
    'Dataset test'  : f'{DATA_DIR}/test/frames',
    'Labels'        : f'{DATA_DIR}/test/test_frame_mask',
}
for name, path in checks.items():
    print('✅' if os.path.exists(path) else '❌', name, '->', path)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Dataset train -> /content/drive/MyDrive/Seminar/UIT-ADrone/train/frames
✅ Dataset test -> /content/drive/MyDrive/Seminar/UIT-ADrone/test/frames
✅ Labels -> /content/drive/MyDrive/Seminar/UIT-ADrone/test/test_frame_mask


## 3. Clone repo & tạo thư mục output

In [ ]:
import sys, os

if not os.path.exists(REPO_DIR):
    !git clone {GITHUB_REPO} {REPO_DIR}
else:
    print('Repo đã tồn tại, pull mới nhất...')
    !cd {REPO_DIR} && git pull

# Tạo thư mục output trên Drive
os.makedirs(f'{SAVE_DIR}/checkpoints', exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
print('Output dir OK:', SAVE_DIR)

# Thêm repo vào sys.path cho cả session
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

!git log --oneline -3

Repo đã tồn tại, pull mới nhất...
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 4 (delta 3), reused 4 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 358 bytes | 358.00 KiB/s, done.
From https://github.com/MinhCYB/TrafficVision-Transformer
   8f170b5..7a48b8c  main       -> origin/main
Updating 8f170b5..7a48b8c
Fast-forward
 scripts/train_baseline_ca.py | 2 +-
 1 file changed, 1 insertion(+), 1 deletion(-)
Output dir OK: /content/drive/MyDrive/Seminar/TrafficVision/experiments_andt_ADrone_baseline_CA
7a48b8c (HEAD -> main, origin/main, origin/HEAD) heh
8f170b5 eiss
5ec161f gluglu


## 4. Cài dependencies

In [ ]:
!pip install timm==1.0.27 scikit-learn tqdm pandas -q

## 5. Kiểm tra import

In [ ]:
from models.vit_baseline_ca import VisionTransformer
from configs.train_config import get_train_config
from data.dataset import DataLoader
print('Import OK ✅')

Import OK ✅


## 6. Train
> ⚠️ Cell này chạy lâu (~vài tiếng). Đừng đóng tab.

In [ ]:
import os
os.environ['SAVE_PATH'] = SAVE_DIR

!PYTHONPATH={REPO_DIR} python scripts/train_baseline_ca.py \
    --train 1 \
    --epochs 2 \
    --batch-size 2 \
    --lr 1e-4 \
    --train-steps 4000 \
    --num-workers 2 \
    --model-arch b16 \
    --image-size 384 \
    --save-dir {SAVE_DIR} \
    --data-dir {DATA_DIR}

 *************************************** 
 The experiment name is anomaly_ste_tte 
 *************************************** 
----------------- Config ---------------
                  anomaly_threshold: None                          
                  attn_dropout_rate: 0.0                           
                         batch_size: 2                             
                     checkpoint_dir: /content/drive/MyDrive/Seminar/TrafficVision/experiments_andt_ADrone_baseline_CA/save/anomaly_ste_tte_UIT-ADrone_bs2_lr0.0001_wd0.0001_260528_050049/checkpoints/
                    checkpoint_path: ./experiments_andt_ADrone_STE_TTE/checkpoints/best.pth
                           data_dir: /content/drive/MyDrive/Seminar/UIT-ADrone
                            dataset: UIT-ADrone                    
                       dropout_rate: 0.1                           
                            emb_dim: 768                           
                             epochs: 2                  

In [ ]:
# Kiểm tra checkpoint đã lưu vào Drive chưa
import os
ckpt_dir = f'{SAVE_DIR}/checkpoints'
if os.path.exists(ckpt_dir):
    files = os.listdir(ckpt_dir)
    print('✅ Checkpoints:', files if files else 'Trống — train chưa xong')
else:
    print('❌ Không tìm thấy:', ckpt_dir)

✅ Checkpoints: ['resume.pth', 'current.pth', 'best.pth']


## 7. Inference (sinh anomaly scores .npy)

In [ ]:
import os
os.environ['SAVE_PATH'] = SAVE_DIR

!PYTHONPATH={REPO_DIR} python scripts/train_baseline_ca.py \
    --train 0 \
    --model-arch b16 \
    --image-size 384 \
    --checkpoint-path {SAVE_DIR}/checkpoints/best.pth \
    --save-dir {SAVE_DIR} \
    --data-dir {DATA_DIR}

 *************************************** 
 The experiment name is anomaly_ste_tte 
 *************************************** 
----------------- Config ---------------
                  anomaly_threshold: None                          
                  attn_dropout_rate: 0.0                           
                         batch_size: 8                             
                     checkpoint_dir: /content/drive/MyDrive/Seminar/TrafficVision/experiments_andt_ADrone_baseline_CA/save/anomaly_ste_tte_UIT-ADrone_bs8_lr0.0001_wd0.0001_260604_104309/checkpoints/
                    checkpoint_path: /content/drive/MyDrive/Seminar/TrafficVision/experiments_andt_ADrone_baseline_CA/checkpoints/best.pth
                           data_dir: /content/drive/MyDrive/Seminar/UIT-ADrone
                            dataset: UIT-ADrone                    
                       dropout_rate: 0.1                           
                            emb_dim: 768                           
         

In [ ]:
# Kiểm tra file .npy đầu ra
import glob, os
npy_files = glob.glob(f'{SAVE_DIR}/*.npy')
print(f'Số file .npy: {len(npy_files)}')
for f in npy_files:
    print(' ', os.path.basename(f))

## 8. Tính AUC-ROC

In [ ]:
LABELS_DIR = f'{DATA_DIR}/test/test_frame_mask/'
ROC_OUT    = f'{RESULTS_DIR}/roc_baseline.png'

!python compute_auc.py \
    --scores-dir {SAVE_DIR}/ \
    --labels-dir {LABELS_DIR} \
    --plot \
    --save-plot {ROC_OUT}

## 9. Visualize ROC comparison
> Chạy cell này **sau khi Người 2 đã xong** inference STE-TTE

In [ ]:
import os
STE_TTE_DIR  = f'{SEMINAR_DIR}/TrafficVision/experiments_andt_ADrone_STE_TTE'
BASELINE_DIR = SAVE_DIR
LABELS_DIR   = f'{DATA_DIR}/test/test_frame_mask/'

print('STE-TTE scores  :', '✅' if os.path.exists(STE_TTE_DIR) else '❌ Chờ Người 2')
print('Baseline scores :', '✅' if os.path.exists(BASELINE_DIR) else '❌')

In [ ]:
from results.visualize import load_all_scores, plot_roc_comparison

ste_scores,      labels = load_all_scores(STE_TTE_DIR, LABELS_DIR)
baseline_scores,      _ = load_all_scores(BASELINE_DIR, LABELS_DIR)

ROC_COMPARE_OUT = f'{RESULTS_DIR}/roc_comparison.png'
plot_roc_comparison(
    models={
        'Baseline CA':        baseline_scores,
        'STE-TTE (Proposed)': ste_scores,
    },
    labels=labels,
    save_path=ROC_COMPARE_OUT
)

from IPython.display import Image
Image(ROC_COMPARE_OUT)

## 10. Backup kết quả về Drive

In [ ]:
import shutil, os
os.makedirs(RESULTS_DIR, exist_ok=True)

files_to_backup = [
    (f'{REPO_DIR}/training_log_baseline.txt', f'{RESULTS_DIR}/training_log_baseline.txt'),
    (f'{REPO_DIR}/results/roc_baseline.png',  f'{RESULTS_DIR}/roc_baseline.png'),
    (f'{REPO_DIR}/results/roc_comparison.png',f'{RESULTS_DIR}/roc_comparison.png'),
]

for src, dst in files_to_backup:
    if os.path.exists(src):
        shutil.copy(src, dst)
        print('✅ Backed up:', os.path.basename(src))
    else:
        print('⏭️  Chưa có:', os.path.basename(src))